In [1]:
%%writefile load_binary.cpp
#include <iostream>
#include <fstream>
#include <vector>
#include <string>

// Instantly loads millions of floats directly into memory
std::vector<float> load_binary_data(const std::string& filepath) {
    std::ifstream file(filepath, std::ios::binary | std::ios::ate);

    if (!file.is_open()) {
        std::cerr << "CRITICAL ERROR: Could not open " << filepath << "\n";
        return std::vector<float>();
    }

    std::streamsize size_in_bytes = file.tellg();
    file.seekg(0, std::ios::beg);

    int num_floats = size_in_bytes / sizeof(float);
    std::vector<float> flattened_data(num_floats);

    if (file.read(reinterpret_cast<char*>(flattened_data.data()), size_in_bytes)) {
        std::cout << "SUCCESS: Loaded " << num_floats << " floats instantly.\n";
    } else {
        std::cerr << "ERROR: Failed to read binary data.\n";
    }

    file.close();
    return flattened_data;
}

int main() {
    // Note: You must actually upload a file named 'test_data.bin' to your Colab session first!
    std::string filepath = "X_sample.bin";
    std::vector<float> my_data = load_binary_data(filepath);
    return 0;
}

Writing load_binary.cpp


In [2]:
!g++ load_binary.cpp -o load_binary -O3

In [3]:
!./load_binary

'.' is not recognized as an internal or external command,
operable program or batch file.


In [5]:
%%bash
# 1. Nuke the broken folder


# 2. Force download the headers (No version checking)
echo "Downloading XGBoost headers..."
git clone --recursive https://github.com/dmlc/xgboost

echo "SUCCESS: Headers downloaded!"

<3>WSL (11 - Relay) ERROR: CreateProcessCommon:640: execvpe(/bin/bash) failed: No such file or directory


CalledProcessError: Command 'b'# 1. Nuke the broken folder\n\n\n# 2. Force download the headers (No version checking)\necho "Downloading XGBoost headers..."\ngit clone --recursive https://github.com/dmlc/xgboost\n\necho "SUCCESS: Headers downloaded!"\n'' returned non-zero exit status 1.

In [ ]:
!git clone --recursive https://github.com/dmlc/xgboost
!cd xgboost && mkdir build && cd build && cmake .. && make -j4

fatal: destination path 'xgboost' already exists and is not an empty directory.
-- The CXX compiler identification is GNU 11.4.0
-- The C compiler identification is GNU 11.4.0
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- CMake version 3.31.10
-- Performing Test XGBOOST_MM_PREFETCH_PRESENT
-- Performing Test XGBOOST_MM_PREFETCH_PRESENT - Success
-- Performing Test XGBOOST_BUILTIN_PREFETCH_PRESENT
-- Performing Test XGBOOST_BUILTIN_PREFETCH_PRESENT - Success
-- xgboost VERSION: 3.3.0
-- Setting build type to 'Release' as none was specified.
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - 

In [ ]:
%%writefile load_test.cpp
#include <iostream>
#include <xgboost/c_api.h>

int main() {
    BoosterHandle booster;

    std::cout << "1. Attempting to create XGBoost Booster...\n";
    if (XGBoosterCreate(NULL, 0, &booster) != 0) {
        std::cerr << "CRITICAL: Failed to create booster.\n";
        return 1;
    }

    std::cout << "2. Attempting to load dummy_model.json...\n";
    if (XGBoosterLoadModel(booster, "xgboost_ids_model2.json") != 0) {
        std::cerr << "CRITICAL: Failed to load model file.\n";
        return 1;
    }

    std::cout << "3. SUCCESS: Model loaded into C++ memory perfectly!\n";

    // Clean up
    XGBoosterFree(booster);

    return 0;
}

Writing load_test.cpp


In [ ]:
%%bash
# Find Python's hidden pre-compiled XGBoost library
LIB_DIR=$(python -c "import xgboost, os; print(os.path.join(os.path.dirname(xgboost.__file__), 'lib'))")

# Compile!
g++ load_test.cpp -Ixgboost/include -Ixgboost/dmlc-core/include -L$LIB_DIR -lxgboost -o load_test

# Execute!
export LD_LIBRARY_PATH=$LD_LIBRARY_PATH:$LIB_DIR
./load_test

1. Attempting to create XGBoost Booster...
2. Attempting to load dummy_model.json...
3. SUCCESS: Model loaded into C++ memory perfectly!


In [ ]:
%%writefile run_real_model.cpp
#include <iostream>
#include <fstream>
#include <vector>
#include <string>
#include <algorithm> // for std::min
#include <xgboost/c_api.h>

int main() {
    // --- YOUR ACTUAL MODEL SPECS ---
    int num_features = 39;
    int num_classes = 6;
    std::string model_path = "xgboost_ids_model2.json";
    std::string data_path = "X_sample.bin";

    // EXACT order from your Python LabelEncoder
    std::vector<std::string> labels = {
        "Benign",
        "Bot",
        "Brute Force",
        "DDoS",
        "DoS",
        "Web Attack"
    };

    // ---------------------------------------------------------
    // 1. LOAD THE REAL MODEL
    // ---------------------------------------------------------
    std::cout << "Loading real model from: " << model_path << "...\n";
    BoosterHandle booster;
    XGBoosterCreate(NULL, 0, &booster);
    if (XGBoosterLoadModel(booster, model_path.c_str()) != 0) {
        std::cerr << "CRITICAL: Failed to load " << model_path << "\n";
        return 1;
    }

    // ---------------------------------------------------------
    // 2. DYNAMICALLY READ THE BINARY FILE
    // ---------------------------------------------------------
    std::ifstream file(data_path, std::ios::binary | std::ios::ate);
    if (!file.is_open()) {
        std::cerr << "CRITICAL: Could not open " << data_path << "\n";
        return 1;
    }

    std::streamsize size_in_bytes = file.tellg();
    file.seekg(0, std::ios::beg);

    int total_floats = size_in_bytes / sizeof(float);
    int num_rows = total_floats / num_features;

    std::cout << "Detected " << num_rows << " network packets with " << num_features << " features.\n";

    std::vector<float> features(total_floats);
    file.read(reinterpret_cast<char*>(features.data()), size_in_bytes);
    file.close();

    // ---------------------------------------------------------
    // 3. CONVERT TO DMATRIX AND PREDICT
    // ---------------------------------------------------------
    DMatrixHandle dmatrix;
    std::cout << "Formatting data for XGBoost...\n";
    XGDMatrixCreateFromMat(features.data(), num_rows, num_features, -1.0f, &dmatrix);

    bst_ulong out_len;
    const float* out_result;

    std::cout << "Executing Multi-Class Inference...\n";
    if (XGBoosterPredict(booster, dmatrix, 0, 0, 0, &out_len, &out_result) != 0) {
        std::cerr << "CRITICAL: Prediction failed.\n";
        return 1;
    }

    // ---------------------------------------------------------
    // 4. PARSE MULTI-CLASS RESULTS (ArgMax)
    // ---------------------------------------------------------
    std::cout << "\n=== INTRUSION DETECTION RESULTS (Showing first 15) ===\n";

    // out_len is total probabilities (num_rows * 6).
    // We only want to loop through the rows.
    int predicted_rows = out_len / num_classes;
    int rows_to_print = std::min(predicted_rows, 15);

    for (int i = 0; i < rows_to_print; i++) {
        int start_index = i * num_classes;

        // Find the ArgMax (Highest Probability) for this specific packet
        int best_class_id = 0;
        float max_probability = out_result[start_index];

        for (int c = 1; c < num_classes; c++) {
            if (out_result[start_index + c] > max_probability) {
                max_probability = out_result[start_index + c];
                best_class_id = c;
            }
        }

        // Formatting to make the console output look clean
        std::string alert = (best_class_id == 0) ? "[OK]" : "[ALERT]";

        std::cout << "Packet " << i + 1
                  << " | " << alert
                  << " Class: " << labels[best_class_id]
                  << " | Confidence: " << (max_probability * 100.0f) << "%\n";
    }

    // ---------------------------------------------------------
    // 5. CLEAN UP MEMORY
    // ---------------------------------------------------------
    XGDMatrixFree(dmatrix);
    XGBoosterFree(booster);

    std::cout << "\nSUCCESS: Processed " << num_rows << " rows perfectly.\n";
    return 0;
}

Overwriting run_real_model.cpp


In [ ]:
%%bash
# Find the library directory
LIB_DIR=$(python -c "import xgboost, os; print(os.path.join(os.path.dirname(xgboost.__file__), 'lib'))")

# Compile the new code
g++ run_real_model.cpp -Ixgboost/include -Ixgboost/dmlc-core/include -L$LIB_DIR -lxgboost -o run_real_model -O3

# Execute
export LD_LIBRARY_PATH=$LD_LIBRARY_PATH:$LIB_DIR
./run_real_model

Loading real model from: xgboost_ids_model2.json...
Detected 100000 network packets with 39 features.
Formatting data for XGBoost...
Executing Multi-Class Inference...

=== INTRUSION DETECTION RESULTS (Showing first 15) ===
Packet 1 | [OK] Class: Benign | Confidence: 99.9944%
Packet 2 | [OK] Class: Benign | Confidence: 99.9789%
Packet 3 | [OK] Class: Benign | Confidence: 99.9944%
Packet 4 | [OK] Class: Benign | Confidence: 99.9942%
Packet 5 | [OK] Class: Benign | Confidence: 99.9944%
Packet 6 | [OK] Class: Benign | Confidence: 99.9942%
Packet 7 | [ALERT] Class: DDoS | Confidence: 99.9915%
Packet 8 | [OK] Class: Benign | Confidence: 99.9943%
Packet 9 | [ALERT] Class: DDoS | Confidence: 99.9912%
Packet 10 | [OK] Class: Benign | Confidence: 99.9925%
Packet 11 | [OK] Class: Benign | Confidence: 99.9371%
Packet 12 | [OK] Class: Benign | Confidence: 99.9943%
Packet 13 | [OK] Class: Benign | Confidence: 99.9937%
Packet 14 | [OK] Class: Benign | Confidence: 99.9935%
Packet 15 | [OK] Class: Ben

In [ ]:
%%writefile mpi_nids.cpp
#include <iostream>
#include <fstream>
#include <vector>
#include <string>
#include <mpi.h>
#include <xgboost/c_api.h>

int main(int argc, char** argv) {
    MPI_Init(&argc, &argv);

    int rank, world_size;
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &world_size);

    int num_features = 39;
    int num_classes = 6;
    std::vector<std::string> labels = {"Benign", "Bot", "Brute Force", "DDoS", "DoS", "Web Attack"};

    BoosterHandle booster;
    XGBoosterCreate(NULL, 0, &booster);
    if (XGBoosterLoadModel(booster, "xgboost_ids_model2.json") != 0) {
        if (rank == 0) std::cerr << "CRITICAL: Failed to load model.\n";
        MPI_Abort(MPI_COMM_WORLD, 1);
    }

    int local_rows = 0;
    std::vector<float> local_data;
    int total_rows = 0;
    int base_rows_per_worker = 0;
    std::vector<float> global_data;

    // --- PHASE 1: SETUP ---
    if (rank == 0) {
        std::ifstream file("X_full.bin", std::ios::binary | std::ios::ate);
        if (!file.is_open()) {
            std::cerr << "[MASTER] Could not open X_full.bin\n";
            MPI_Abort(MPI_COMM_WORLD, 1);
        }

        std::streamsize size_in_bytes = file.tellg();
        file.seekg(0, std::ios::beg);

        int total_floats = size_in_bytes / sizeof(float);
        total_rows = total_floats / num_features;

        global_data.resize(total_floats);
        file.read(reinterpret_cast<char*>(global_data.data()), size_in_bytes);
        file.close();

        base_rows_per_worker = total_rows / world_size;
        int remainder = total_rows % world_size;
        local_rows = base_rows_per_worker + remainder;
        local_data.assign(global_data.begin(), global_data.begin() + (local_rows * num_features));
    }

    // =========================================================
    // ⏱️ START THE STOPWATCH
    // =========================================================
    // MPI_Barrier forces all nodes to wait here until everyone is ready.
    MPI_Barrier(MPI_COMM_WORLD);

    // Master node clicks the stopwatch button
    double start_time = 0.0;
    if (rank == 0) {
        start_time = MPI_Wtime();
        std::cout << "[MASTER] Stopwatch started. Distributing data to Workers...\n";
    }

    // --- PHASE 2: DISTRIBUTION ---
    if (rank == 0) {
        int offset = local_rows;
        for (int i = 1; i < world_size; i++) {
            MPI_Send(&base_rows_per_worker, 1, MPI_INT, i, 0, MPI_COMM_WORLD);
            MPI_Send(&global_data[offset * num_features], base_rows_per_worker * num_features, MPI_FLOAT, i, 1, MPI_COMM_WORLD);
            offset += base_rows_per_worker;
        }
    } else {
        MPI_Recv(&local_rows, 1, MPI_INT, 0, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
        local_data.resize(local_rows * num_features);
        MPI_Recv(local_data.data(), local_rows * num_features, MPI_FLOAT, 0, 1, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
    }

    // --- PHASE 3: PARALLEL INFERENCE ---
    DMatrixHandle dmatrix;
    XGDMatrixCreateFromMat(local_data.data(), local_rows, num_features, -1.0f, &dmatrix);

    bst_ulong out_len;
    const float* out_result;
    XGBoosterPredict(booster, dmatrix, 0, 0, 0, &out_len, &out_result);

    std::vector<int> local_predictions(local_rows);
    for (int i = 0; i < local_rows; i++) {
        int start_idx = i * num_classes;
        int best_class = 0;
        float max_prob = out_result[start_idx];

        for (int c = 1; c < num_classes; c++) {
            if (out_result[start_idx + c] > max_prob) {
                max_prob = out_result[start_idx + c];
                best_class = c;
            }
        }
        local_predictions[i] = best_class;
    }

    // --- PHASE 4: GATHERING RESULTS ---
    if (rank == 0) {
        std::vector<int> global_predictions(total_rows);
        std::copy(local_predictions.begin(), local_predictions.end(), global_predictions.begin());
        int offset = local_rows;

        for (int i = 1; i < world_size; i++) {
            MPI_Recv(&global_predictions[offset], base_rows_per_worker, MPI_INT, i, 2, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
            offset += base_rows_per_worker;
        }

        // =========================================================
        // 🛑 STOP THE STOPWATCH
        // =========================================================
        double end_time = MPI_Wtime();
        double total_time = end_time - start_time;

        std::cout << "\n==================================================================\n";
        std::cout << "🚀 PARALLEL PERFORMANCE METRICS\n";
        std::cout << "==================================================================\n";
        std::cout << "Total Packets Processed : " << total_rows << "\n";
        std::cout << "MPI Processes Used      : " << world_size << "\n";
        std::cout << "Total Execution Time    : " << total_time << " seconds\n";
        std::cout << "Throughput              : " << (total_rows / total_time) << " packets per second\n";
        std::cout << "==================================================================\n";

    } else {
        MPI_Send(local_predictions.data(), local_rows, MPI_INT, 0, 2, MPI_COMM_WORLD);
    }

    // --- CLEANUP ---
    XGDMatrixFree(dmatrix);
    XGBoosterFree(booster);
    MPI_Finalize();
    return 0;
}

Overwriting mpi_nids.cpp


In [ ]:
%%bash
# Find the library directory
LIB_DIR=$(python -c "import xgboost, os; print(os.path.join(os.path.dirname(xgboost.__file__), 'lib'))")

# Compile using MPI
mpicxx mpi_nids.cpp -Ixgboost/include -Ixgboost/dmlc-core/include -L$LIB_DIR -lxgboost -o mpi_nids -O3

# Execute using 4 parallel MPI processes, forcing Colab to oversubscribe its 2 cores
export LD_LIBRARY_PATH=$LD_LIBRARY_PATH:$LIB_DIR
mpirun --allow-run-as-root --oversubscribe -np 4 ./mpi_nids

[MASTER] Stopwatch started. Distributing data to Workers...

🚀 PARALLEL PERFORMANCE METRICS
Total Packets Processed : 403298
MPI Processes Used      : 4
Total Execution Time    : 6.57316 seconds
Throughput              : 61355.3 packets per second


In [ ]:
%%writefile serial_nids.cpp
#include <iostream>
#include <fstream>
#include <vector>
#include <string>
#include <chrono> // Standard C++ high-resolution stopwatch
#include <xgboost/c_api.h>

int main() {
    int num_features = 39;
    int num_classes = 6;
    std::vector<std::string> labels = {"Benign", "Bot", "Brute Force", "DDoS", "DoS", "Web Attack"};

    // ---------------------------------------------------------
    // 1. SETUP & LOAD MODEL
    // ---------------------------------------------------------
    BoosterHandle booster;
    XGBoosterCreate(NULL, 0, &booster);
    if (XGBoosterLoadModel(booster, "xgboost_ids_model2.json") != 0) {
        std::cerr << "CRITICAL: Failed to load model.\n";
        return 1;
    }

    // CRITICAL FOR BASELINE: Force XGBoost to use exactly 1 thread.
    // If you don't do this, XGBoost will cheat and use all cores!
    XGBoosterSetParam(booster, "nthread", "1");

    // ---------------------------------------------------------
    // 2. READ THE BINARY FILE
    // ---------------------------------------------------------
    std::ifstream file("X_full.bin", std::ios::binary | std::ios::ate);
    if (!file.is_open()) {
        std::cerr << "CRITICAL: Could not open X_ful.bin\n";
        return 1;
    }

    std::streamsize size_in_bytes = file.tellg();
    file.seekg(0, std::ios::beg);

    int total_floats = size_in_bytes / sizeof(float);
    int total_rows = total_floats / num_features;

    std::vector<float> data(total_floats);
    file.read(reinterpret_cast<char*>(data.data()), size_in_bytes);
    file.close();

    std::vector<int> predictions(total_rows);

    // =========================================================
    // ⏱️ START THE STOPWATCH
    // =========================================================
    std::cout << "Stopwatch started. Launching PURE SERIAL inference...\n";
    auto start_time = std::chrono::high_resolution_clock::now();

    // ---------------------------------------------------------
    // 3. SERIAL INFERENCE & ARGMAX LOOP
    // ---------------------------------------------------------
    DMatrixHandle dmatrix;
    XGDMatrixCreateFromMat(data.data(), total_rows, num_features, -1.0f, &dmatrix);

    bst_ulong out_len;
    const float* out_result;

    // Predict everything on 1 core
    XGBoosterPredict(booster, dmatrix, 0, 0, 0, &out_len, &out_result);

    // Find the ArgMax for all rows on 1 core
    for (int i = 0; i < total_rows; i++) {
        int start_idx = i * num_classes;
        int best_class = 0;
        float max_prob = out_result[start_idx];

        for (int c = 1; c < num_classes; c++) {
            if (out_result[start_idx + c] > max_prob) {
                max_prob = out_result[start_idx + c];
                best_class = c;
            }
        }
        predictions[i] = best_class;
    }

    // =========================================================
    // 🛑 STOP THE STOPWATCH
    // =========================================================
    auto end_time = std::chrono::high_resolution_clock::now();
    std::chrono::duration<double> total_time = end_time - start_time;

    // ---------------------------------------------------------
    // 4. PRINT METRICS
    // ---------------------------------------------------------
    std::cout << "\n==================================================================\n";
    std::cout << "🚀 STRICT SERIAL PERFORMANCE METRICS (BASELINE)\n";
    std::cout << "==================================================================\n";
    std::cout << "Total Packets Processed : " << total_rows << "\n";
    std::cout << "Threads Used            : 1 (Pure Sequential)\n";
    std::cout << "Total Execution Time    : " << total_time.count() << " seconds\n";
    std::cout << "Throughput              : " << (total_rows / total_time.count()) << " packets per second\n";
    std::cout << "==================================================================\n";

    XGDMatrixFree(dmatrix);
    XGBoosterFree(booster);
    return 0;
}

Overwriting serial_nids.cpp


In [ ]:
%%bash
# Find the library directory
LIB_DIR=$(python -c "import xgboost, os; print(os.path.join(os.path.dirname(xgboost.__file__), 'lib'))")

# Compile using standard g++
g++ serial_nids.cpp -Ixgboost/include -Ixgboost/dmlc-core/include -L$LIB_DIR -lxgboost -o serial_nids -O3

# Execute directly
export LD_LIBRARY_PATH=$LD_LIBRARY_PATH:$LIB_DIR
./serial_nids

Stopwatch started. Launching PURE SERIAL inference...

🚀 STRICT SERIAL PERFORMANCE METRICS (BASELINE)
Total Packets Processed : 322638
Threads Used            : 1 (Pure Sequential)
Total Execution Time    : 4.76286 seconds
Throughput              : 67740.4 packets per second
